# 2.4 — Incremental Implementation

---

## The problem with the naive average

$$Q_n \;\doteq\; \frac{R_1 + R_2 + \cdots + R_{n-1}}{n-1}$$

Computed literally, this needs memory that grows with $n$ and $O(n)$ work per step.
Unacceptable for a method meant to run forever.

## The incremental form

$$
\begin{aligned}
Q_{n+1} &= \frac{1}{n}\sum_{i=1}^{n} R_i
= \frac{1}{n}\left(R_n + \sum_{i=1}^{n-1} R_i\right)
= \frac{1}{n}\Big(R_n + (n-1)\,Q_n\Big) \\[4pt]
&= \frac{1}{n}\Big(R_n + nQ_n - Q_n\Big) \\[4pt]
&= \boxed{\;Q_n + \frac{1}{n}\big[R_n - Q_n\big]\;}
\end{aligned}
$$

Constant memory, constant time. But the algebraic convenience is the least interesting
thing about it.

## The master update rule

$$\text{NewEstimate} \;\leftarrow\; \text{OldEstimate} \;+\; \text{StepSize}\,\Big[\,\text{Target} - \text{OldEstimate}\,\Big]$$

Read $[\text{Target} - \text{OldEstimate}]$ as an **error**. You move your estimate a
fraction of the way toward the target. This pattern — with different targets and
different step sizes — is essentially every learning rule in this book: TD learning,
Q-learning, SARSA, policy gradients. Sample-average is the special case
$\alpha_n = 1/n$.

## The complete algorithm

```
Initialise, for a = 1..k:
    Q(a) <- 0
    N(a) <- 0

Loop forever:
    A <- argmax_a Q(a)   with probability 1 - eps
         random action   with probability eps
    R <- bandit(A)
    N(A) <- N(A) + 1
    Q(A) <- Q(A) + (1/N(A)) [R - Q(A)]
```

In [ ]:
import sys, os
sys.path.insert(0, os.getcwd())

import numpy as np
import matplotlib.pyplot as plt
from bandit_utils import (Testbed, run_bandit, plot_pair, argmax_random_tiebreak,
                          hide, banner, RUNS, STEPS)

plt.rcParams["figure.dpi"] = 110
banner()

## Verify the identity numerically

Not because you doubt the algebra, but because getting into the habit of checking
update rules against a brute-force reference will save you many hours in later chapters.

In [ ]:
rng = np.random.default_rng(0)
R = rng.normal(1.5, 2.0, size=200)

naive = np.array([R[:n].mean() for n in range(1, len(R) + 1)])

incremental = np.zeros(len(R))
Q = 0.0
for n, r in enumerate(R, start=1):
    Q += (r - Q) / n
    incremental[n - 1] = Q

print("max abs difference:", np.abs(naive - incremental).max())
assert np.allclose(naive, incremental)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(naive, lw=3, alpha=0.4, label="naive $\\frac{1}{n}\\sum R_i$")
ax.plot(incremental, ls="--", color="crimson", label="incremental $Q + \\frac{1}{n}[R-Q]$")
ax.axhline(1.5, color="k", ls=":", label="true mean")
ax.set_xlabel("n"); ax.set_ylabel("$Q_n$"); ax.legend(); ax.grid(alpha=0.3)
ax.set_title("Identical, but one uses O(1) memory")
plt.show()

## What the step size is *doing*

Write the update as a weighted blend:

$$Q_{n+1} = (1 - \alpha_n)\,Q_n \;+\; \alpha_n R_n$$

So $\alpha_n$ is literally "how much do I trust this new sample relative to everything
I already believe". With $\alpha_n = 1/n$:

- $n=1$: $\alpha = 1$ — the first sample completely overwrites the initial estimate.
  (This is why $Q_1(a)$ has no effect on sample-average methods after the first pull of
  $a$ — a fact Section 2.6 exploits.)
- $n=100$: $\alpha = 0.01$ — new evidence barely moves the needle.

The estimate becomes progressively more inert. That is exactly right if $q_*$ is
constant, and exactly wrong if it is not.

---

### Predict first

Unroll the $1/n$ update all the way back to $Q_1$. Each past reward $R_i$
ends up with some weight in $Q_{n+1}$.

1. What weight does $R_1$ carry in $Q_{101}$? What about $R_{100}$?
2. Now suppose $\alpha$ is a **constant** 0.1 instead. Same question.

*Write your guess down (mentally or in the cell below) before running the next cell. The point is not to be right — it is to make the surprise informative when you are wrong.*

In [ ]:
def weights(n, alpha=None):
    '''Weight each past reward R_1..R_n receives in Q_{n+1}.'''
    w = np.zeros(n)
    for i in range(1, n + 1):
        a_i = alpha if alpha else 1.0 / i
        w[i - 1] = a_i
        w[:i - 1] *= (1 - a_i)
    return w

n = 100
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
for alpha, lab in [(None, "$\\alpha_n = 1/n$ (sample average)"),
                   (0.1, "$\\alpha = 0.1$ (constant)"),
                   (0.3, "$\\alpha = 0.3$ (constant)")]:
    w = weights(n, alpha)
    axes[0].plot(np.arange(1, n + 1), w, label=lab)
    axes[1].semilogy(np.arange(1, n + 1), np.maximum(w, 1e-12), label=lab)
for ax in axes:
    ax.set_xlabel("i  (which past reward)")
    ax.set_ylabel("weight of $R_i$ in $Q_{101}$")
    ax.legend(fontsize=9); ax.grid(alpha=0.3)
axes[1].set_title("log scale: constant alpha decays geometrically")
axes[0].set_title("Every past reward weighted equally vs. recency-weighted")
plt.tight_layout(); plt.show()

for alpha in [None, 0.1]:
    w = weights(100, alpha)
    lab = "1/n" if alpha is None else f"alpha={alpha}"
    print(f"{lab:<10} w(R_1)={w[0]:.5f}  w(R_100)={w[-1]:.5f}  sum={w.sum():.5f}")

In [ ]:
hide('''With <b>1/n</b> every reward gets weight exactly 1/n = 0.01 &mdash; a flat line.
That is what &quot;average&quot; means: all evidence counts the same regardless of age.
<br><br>With <b>constant &alpha;=0.1</b>, R_100 gets 0.1 and R_1 gets
0.1 &times; 0.9<sup>99</sup> &asymp; 3&times;10<sup>-6</sup> &mdash; essentially nothing.
The weights decay <i>exponentially</i> with age. The estimate has an effective memory of
roughly 1/&alpha; = 10 samples.
<br><br>Also note both sets of weights sum to (almost) 1 &mdash; constant &alpha; misses
by (1-&alpha;)<sup>n</sup>, which is the leftover weight still sitting on the initial
estimate Q_1. Section 2.6 is built entirely on that leftover term.''')

## Convergence conditions (worth knowing now)

The standard stochastic-approximation conditions for $Q_n \to q_*$ with probability 1:

$$\sum_{n=1}^{\infty} \alpha_n(a) = \infty \qquad\text{and}\qquad \sum_{n=1}^{\infty} \alpha_n^2(a) < \infty$$

- The **first** says the steps are collectively large enough to overcome any initial
  condition and any run of bad luck.
- The **second** says they eventually shrink enough to converge rather than oscillate.

$\alpha_n = 1/n$ satisfies both ($\sum 1/n$ diverges, $\sum 1/n^2 = \pi^2/6$).
Constant $\alpha$ satisfies the first but **not** the second — so it never converges,
it keeps fluctuating in response to the most recent rewards. Section 2.5 argues that
this apparent defect is exactly what you want in a changing world.

In [ ]:
n = np.arange(1, 100001)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for a_n, lab in [(1/n, "$1/n$"), (np.full_like(n, 0.1, dtype=float), "$0.1$"),
                 (1/n**0.7, "$n^{-0.7}$"), (1/n**1.2, "$n^{-1.2}$")]:
    axes[0].loglog(n, np.cumsum(a_n), label=lab)
    axes[1].loglog(n, np.cumsum(a_n**2), label=lab)
axes[0].set_title(r"$\sum \alpha_n$  (want $\to \infty$)")
axes[1].set_title(r"$\sum \alpha_n^2$  (want finite)")
for ax in axes:
    ax.set_xlabel("n"); ax.legend(fontsize=9); ax.grid(alpha=0.3, which="both")
plt.tight_layout(); plt.show()

print("n^-1.2 : sum alpha converges -> can stop short of q*, condition 1 violated")
print("n^-0.7 : both conditions satisfied, converges but slowly")
print("alpha=0.1 : sum alpha^2 diverges -> never settles")

Look at $n^{-1.2}$ in the left panel: $\sum\alpha_n$ flattens out. The total distance the
estimate can *ever* travel is bounded, so if it starts far from $q_*$ it may never
arrive. That is what violating condition 1 costs you.

In practice, Sutton & Barto note, sequences satisfying both conditions converge very
slowly and need tuning anyway — so they are rare outside of theory.

## Takeaways for 2.4

1. $Q_{n+1} = Q_n + \frac{1}{n}[R_n - Q_n]$ — $O(1)$ time and memory.
2. The general form *NewEstimate ← OldEstimate + StepSize [Target − OldEstimate]* is the
   backbone of the whole book.
3. Step size = trust in the newest sample. $1/n$ weights all history equally;
   constant $\alpha$ weights it exponentially by recency.
4. Convergence needs $\sum\alpha_n = \infty$, $\sum\alpha_n^2 < \infty$. Constant $\alpha$
   fails the second — deliberately.

**Next:** the case where *not* converging is the right answer — Section 2.5.